# Open-Source Models -- African Language Confusion Sweep

Clean, systematic version of the exploratory `Open_Source_Models.ipynb` blueprint: instead
of hand-typed prompt lists and ad hoc per-model cells, this runs every model in `MODELS`
over the full consolidated prompt set built by `Build_Prompt_Dataset.ipynb`
(`prompts/all_prompts.csv`), and logs results to `outputs/{model_key}.csv` in the same
`id, model, completion, task, source, language` schema the language-confusion repo's
`compute_metrics.py` expects.

**Change from the blueprint's ad hoc cells:** each prompt is sent as an independent
single-turn message, not appended to a growing conversation. The exploratory notebook
built up conversation history across prompts within a language (each new prompt saw every
earlier reply), which confounds language-confusion measurements -- a bad reply on prompt N
could just be inherited context from prompt N-1, not something caused by prompt N. See
`model_clients.py` for the shared sweep/retry/resume logic used by both this notebook and
`Closed_Source_Models.ipynb`.

### Setup

Run once, from the `african-language-confusion/` folder, in a virtual environment (keeps
these dependencies out of your global Python install):

```bash
python -m venv venv
venv\Scriptsctivate      # Windows
source venv/bin/activate    # macOS/Linux
pip install -r requirements.txt
```

Then pick `venv` as this notebook's kernel before running the cells below.


In [2]:
import pandas as pd
from openai import OpenAI

import model_clients as mc

OPENROUTER_API_KEY = mc.get_api_key("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)


### Models

OpenRouter model slugs -- add/remove entries here to change what the sweep covers.

In [3]:
MODELS = {
    "qwen": "qwen/qwen3-235b-a22b-2507",
    "mistral": "mistralai/mistral-large-2512",
    "glm": "zai-org/GLM-5.2",
    "kimi": "moonshotai/Kimi-K2-Instruct-0905",
    "command-r-plus": "cohere/command-r-plus-08-2024",
    "llama": "meta-llama/llama-3.3-70b-instruct",
}


### Chat helper

One independent, single-turn call per prompt (see the design note above).

In [7]:
def make_call_fn(model_id, temperature=0.5, max_tokens=1024):
    def call_fn(prompt):
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        return response.choices[0].message.content
    return call_fn


### Load prompts

In [5]:
all_prompts = pd.read_csv("prompts/all_prompts.csv")
all_prompts.shape


(11774, 5)

### Quick check on a single language before running the full sweep

In [14]:
prompts_preview = mc.get_prompts(all_prompts, language="igbo", source="aya", n=5)
call_fn = make_call_fn(MODELS["qwen"])
for _, row in prompts_preview.iterrows():
    print(f"PROMPT: {row['prompt']}")
    print(f"REPLY:  {call_fn(row['prompt'])}")
    print()


PROMPT: Kedụ Ihe anọ ị ma maka steeti Katsina? 
REPLY:  N'okwu maka steeti Katsina, e nwere ihe anọ dị mkpa ịkwụ ụgwọ n'ịmụ:

1. **Ebe obibi ya na nọchitere ya**:  
   Katsina bụ steeti nke nọ na-agwa n'ala ọdịda dị n'etiti **Nigeria**. Ọ nọ na-agwa **Hausa** na **Fulani**, ma ọ bụrụ na ọ dị n'akụkụ nke **Kano**, **Zamfara**, **Jigawa**, **Bauchi**, **Niger**, **Kaduna**, ma na-agba ọsọ **Niger Republic**.

2. **Ebe obibi ọdịda**:  
   Ọ bụ ebe ọdịda nke **Hausa Kingdoms** ma ọ bụrụ na ọ bụ nke ọma nke ọma n'oge ọdịda. Katsina abụrụrụ obodo ọdịda dị mkpa n'oge a, na-enye ohere maka mmekọrịta na ọdịda na **Sahel**.

3. **Agụmakwụkwọ na mmemme**:  
   Katsina dị na-agba mbọ n'ime mkparịta ụka agụmakwụkwọ dị ka **Ahmadu Bello University (ABU), Zaria**, ma ọ bụrụ na ọ nwere **Katsina State University**. Ọ bụ ebe ọdịda nke **Malami Bello** (onye isi oche Nigeria n'afọ 1979–1983) ma ọ bụrụ na **Umaru Yar’Adua** (onye isi oche Nigeria n'afọ 2007–2010) – ndị e meriri n'ọdịda.

4. **Ọrụ ike na 

### Small end-to-end test batch (Igbo)

A minimal real run -- writes to `outputs/{model_key}.csv` (unlike the print-only preview
above), so `Compute_Metrics.ipynb` has something to score. Good for confirming the whole
pipeline (prompts -> completions -> LPR/WPR) works before committing to a full sweep.

In [8]:
igbo_test_prompts = mc.get_prompts(all_prompts, language="igbo", n=5)
test_call_fn = make_call_fn(MODELS["qwen"])
mc.run_benchmark("qwen", test_call_fn, igbo_test_prompts, output_dir="outputs", delay=1.0)

[qwen] 0 already done, 5 remaining (5 total) -> outputs\qwen.csv
[qwen] done -> outputs\qwen.csv


'outputs\\qwen.csv'

### Full sweep

Runs every model in `MODELS` over every prompt in `all_prompts`, writing/resuming
`outputs/{model_key}.csv`. This is a lot of API calls (18 languages x several sources x
several models) -- if you want to scope it down first, filter `all_prompts` with
`mc.get_prompts(all_prompts, language=[...], source=[...], task=...)` before passing it in.
`delay` adds a small pause between calls to stay under rate limits; raise it if you
hit 429s.

In [ ]:
output_paths = mc.run_sweep(
    MODELS,
    make_call_fn,
    all_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths

### Per-dataset sweeps

Same sweep as above, scoped to one prompt dataset (`source`) at a time. Useful for running/
resuming a single dataset in isolation (e.g. to prioritize it, or re-run after a dataset-
specific fix) instead of the full `all_prompts` set. These write into the same
`outputs/{model_key}.csv` files as the full sweep -- `run_benchmark` skips ids it has
already logged, so running a per-dataset cell and the full sweep cell against the same
`outputs/` directory is safe either order.

#### aya

In [ ]:
aya_prompts = mc.get_prompts(all_prompts, source="aya")
output_paths_aya = mc.run_sweep(
    MODELS,
    make_call_fn,
    aya_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_aya

#### afriqa

In [ ]:
afriqa_prompts = mc.get_prompts(all_prompts, source="afriqa")
output_paths_afriqa = mc.run_sweep(
    MODELS,
    make_call_fn,
    afriqa_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_afriqa

#### polywrite

In [ ]:
polywrite_prompts = mc.get_prompts(all_prompts, source="polywrite")
output_paths_polywrite = mc.run_sweep(
    MODELS,
    make_call_fn,
    polywrite_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_polywrite

#### dolly

In [ ]:
dolly_prompts = mc.get_prompts(all_prompts, source="dolly")
output_paths_dolly = mc.run_sweep(
    MODELS,
    make_call_fn,
    dolly_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_dolly

#### sharegpt

In [ ]:
sharegpt_prompts = mc.get_prompts(all_prompts, source="sharegpt")
output_paths_sharegpt = mc.run_sweep(
    MODELS,
    make_call_fn,
    sharegpt_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_sharegpt

#### okapi

In [ ]:
okapi_prompts = mc.get_prompts(all_prompts, source="okapi")
output_paths_okapi = mc.run_sweep(
    MODELS,
    make_call_fn,
    okapi_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_okapi